In [0]:
from scripts.silver import silver_loader
from scripts.utils import silver_utils

print(dir(silver_loader))
print(dir(silver_utils))

In [0]:
from pyspark.sql.functions import (
    col, when, sum, lit, trim, lower, upper,
    to_date, current_timestamp,round
)
from datetime import datetime
import uuid

In [0]:
df=silver_loader.get_incremental(spark,'categories',"id")

In [0]:
silver_utils.check_schema(df)

In [0]:
df=df.drop("image","slug")

In [0]:
df= silver_utils.cast_types(df,{
    "id" : "long",
    "name" : "string",
    "creationAt" : "timestamp",
    "updatedAt" : "timestamp"})

In [0]:
df=(df.withColumnRenamed("id","category_id") 
    .withColumnRenamed("name","category_name") 
    .withColumnRenamed("creationAt","user_creation_at") 
    .withColumnRenamed("updatedAt","user_updated_at")
    )

In [0]:
silver_utils.null_profiling(df,"categories")

df=silver_utils.handle_nulls_drop(df,["category_id","category_name"])

In [0]:
df=silver_utils.handle_duplicates(df,["category_id"])

In [0]:
df = silver_utils.standardize_strings(
    df,{
    "category_name" : lambda c : lower(trim(c))
    }
)

In [0]:
checks = [
    #nulls
    (
        "category_id",
        "category_id not null",
        col("category_id").isNotNull()
    ),
    (
        "category_name",
        "category_name not null",
        col("category_name").isNotNull()
    ),

    #business rules
    (
       "user_creation_at",
       "user_creation_at must be <= user_updated_at",
       col("user_creation_at") <= col("user_updated_at")
    )
]

dq_table = silver_utils.build_dq_table(
    spark=spark,
    df=df,
    checks=checks,
    table_name="users_api",
    warn_threshold=5.0
)

dq_table.write.mode("append").saveAsTable("workspace.ecommerce_silver.categories_api_dq")
dq_saved = spark.read.table("workspace.ecommerce_silver.categories_api_dq")
display(dq_saved)

In [0]:
df = silver_utils.add_silver_metadata(df)

In [0]:
#Append new rows into silver ,,first time use overwrite afteer use append
df.write.format("delta").mode("append").saveAsTable("workspace.ecommerce_silver.categories")
print(" Data written to silver table successfully")

In [0]:
%sql
--sanity check
SELECT * FROM workspace.ecommerce_silver.categories LIMIT 10